## Realisé par : Mouhammad thahir Ousmane 

## 1-Construire l’index sur l’ensemble du corpus

In [59]:
import glob
import json
import re

def lire_fichier(chemin):
    with open(chemin, 'r', encoding='utf-8') as f:
        return f.read()

def decouper_mots(chaine):
    # Utiliser une expression régulière pour découper les mots en minuscules
    return re.findall(r'\b\w+\b', chaine.lower())

index = {}

# Utiliser le chemin complet
chemin_base = "data/europarl/fr/*"

# Parcourir tous les fichiers du corpus
for chemin in glob.glob(chemin_base):
    chaine = lire_fichier(chemin)
    mots = decouper_mots(chaine)
    for mot in mots:
        if mot not in index:
            index[mot] = set()  # Utiliser un ensemble pour éviter les doublons
        index[mot].add(chemin)

# Afficher la taille du vocabulaire
print(f"Taille du vocabulaire : {len(index)}")

# Vérifier et afficher le nombre de textes contenant des mots spécifiques
mots_a_verifier = ["indique", "européenne", "toto"]
for mot in mots_a_verifier:
    if mot in index:
        print(f"Le mot '{mot}' est présent dans {len(index[mot])} documents.")
    else:
        print(f"Le mot '{mot}' n'est pas présent dans le corpus indexé.")

# Stocker l'index
index = {mot: list(chemins) for mot, chemins in index.items()}  # Convertir les ensembles en listes pour le stockage JSON

with open("index.json", "w") as w:
    w.write(json.dumps(index))

Taille du vocabulaire : 81638
Le mot 'indique' est présent dans 293 documents.
Le mot 'européenne' est présent dans 363 documents.
Le mot 'toto' est présent dans 2 documents.


## 2- Charger l’index et vérifier les mots

In [60]:
import json

def charger_index(fichier):
    try:
        with open(fichier, 'r', encoding='utf-8') as f:
            index = json.load(f)
        print("Index chargé avec succès.")
        return index
    except FileNotFoundError:
        print("Le fichier 'index.json' n'a pas été trouvé.")
        return None
    except json.JSONDecodeError:
        print("Erreur lors du chargement du fichier 'index.json'. Assurez-vous qu'il est correctement formaté.")
        return None

def decouper_requete(requete):
    # Convertir la requête en minuscules
    mots_requete = requete.lower().split()
    print(f"Mots de la requête : {mots_requete}")
    return mots_requete

# Charger l'index
index = charger_index("index.json")

if index:
    requete = "Commission Européenne"
    mots_requete = decouper_requete(requete)

Index chargé avec succès.
Mots de la requête : ['commission', 'européenne']


## 3- Requêter l’index et afficher les résultats

In [61]:
def chercher_documents(index, mots_requete):
    documents_trouves = {}
    for mot in mots_requete:
        if mot in index:
            print(f"Le mot '{mot}' est présent dans l'index et est trouvé dans les documents suivants : {index[mot]}")
            for doc in index[mot]:
                if doc not in documents_trouves:
                    documents_trouves[doc] = 0
                documents_trouves[doc] += 1
        else:
            print(f"Le mot '{mot}' n'est pas présent dans l'index.")
    return documents_trouves

def afficher_documents(documents_trouves):
    if not documents_trouves:
        print("Aucun mot de la requête n'est connu.")
    else:
        documents_trouves_tries = sorted(documents_trouves.items(), key=lambda x: x[1], reverse=True)
        print("Documents trouvés (limité aux 10 premiers) :")
        for doc, count in documents_trouves_tries[:10]:
            print(f"{doc} (contient {count} mots de la requête)")

if index:
    documents_trouves = chercher_documents(index, mots_requete)
    afficher_documents(documents_trouves)

Le mot 'commission' est présent dans l'index et est trouvé dans les documents suivants : ['data/europarl/fr/ep-04-07-22-fr.txt', 'data/europarl/fr/ep-01-09-06-fr.txt', 'data/europarl/fr/ep-00-05-03-fr.txt', 'data/europarl/fr/ep-02-05-14-fr.txt', 'data/europarl/fr/ep-02-03-11-fr.txt', 'data/europarl/fr/ep-03-10-08-fr.txt', 'data/europarl/fr/ep-03-06-03-fr.txt', 'data/europarl/fr/ep-04-12-16-fr.txt', 'data/europarl/fr/ep-02-11-20-fr.txt', 'data/europarl/fr/ep-03-06-19-fr.txt', 'data/europarl/fr/ep-01-12-17-fr.txt', 'data/europarl/fr/ep-02-04-24-fr.txt', 'data/europarl/fr/ep-03-04-08-fr.txt', 'data/europarl/fr/ep-01-10-23-fr.txt', 'data/europarl/fr/ep-02-11-18-fr.txt', 'data/europarl/fr/ep-00-06-13-fr.txt', 'data/europarl/fr/ep-02-03-20-fr.txt', 'data/europarl/fr/ep-00-04-10-fr.txt', 'data/europarl/fr/ep-05-05-25-fr.txt', 'data/europarl/fr/ep-05-11-17-fr.txt', 'data/europarl/fr/ep-01-07-02-fr.txt', 'data/europarl/fr/ep-04-10-27-fr.txt', 'data/europarl/fr/ep-02-05-29-fr.txt', 'data/europar

## 4- Ajouter et intégrer la fonction afficher_contextes

In [63]:
import re

def afficher_contextes(chaine, terme, taille_contexte=30):
    matches = re.finditer(re.escape(terme), chaine)
    contexts = []
    for match in matches:
        gauche = max(match.start() - taille_contexte, 0)
        droite = match.end() + taille_contexte
        contexts.append(chaine[gauche:droite])
    return contexts

def chercher_documents_et_contextes(index, mots_requete, taille_contexte=30):
    documents_trouves = {}
    contextes = {}
    for mot in mots_requete:
        if mot in index:
            print(f"Le mot '{mot}' est présent dans l'index et est trouvé dans les documents suivants : {index[mot]}")
            for doc in index[mot]:
                if doc not in documents_trouves:
                    documents_trouves[doc] = 0
                    contextes[doc] = {}
                documents_trouves[doc] += 1
                # Lire le fichier et trouver les contextes
                chaine = lire_fichier(doc)
                if mot not in contextes[doc]:
                    contextes[doc][mot] = []
                contextes[doc][mot].extend(afficher_contextes(chaine, mot, taille_contexte))
        else:
            print(f"Le mot '{mot}' n'est pas présent dans l'index.")
    return documents_trouves, contextes

def afficher_contextes_documents(contextes):
    if not contextes:
        print("Aucun contexte trouvé pour les mots de la requête.")
    else:
        print("Contextes trouvés :")
        for doc, contextes_mots in contextes.items():
            print(f"\nDocument: {doc}")
            for mot, contextes_list in contextes_mots.items():
                print(f"\nMot: {mot}")
                for contexte in contextes_list:
                    print(f"...{contexte}...")

# Exemple d'utilisation
if index:
    requete = "Commission Européenne"
    mots_requete = decouper_requete(requete)
    documents_trouves, contextes = chercher_documents_et_contextes(index, mots_requete)
    afficher_contextes_documents(contextes)

Mots de la requête : ['commission', 'européenne']
Le mot 'commission' est présent dans l'index et est trouvé dans les documents suivants : ['data/europarl/fr/ep-04-07-22-fr.txt', 'data/europarl/fr/ep-01-09-06-fr.txt', 'data/europarl/fr/ep-00-05-03-fr.txt', 'data/europarl/fr/ep-02-05-14-fr.txt', 'data/europarl/fr/ep-02-03-11-fr.txt', 'data/europarl/fr/ep-03-10-08-fr.txt', 'data/europarl/fr/ep-03-06-03-fr.txt', 'data/europarl/fr/ep-04-12-16-fr.txt', 'data/europarl/fr/ep-02-11-20-fr.txt', 'data/europarl/fr/ep-03-06-19-fr.txt', 'data/europarl/fr/ep-01-12-17-fr.txt', 'data/europarl/fr/ep-02-04-24-fr.txt', 'data/europarl/fr/ep-03-04-08-fr.txt', 'data/europarl/fr/ep-01-10-23-fr.txt', 'data/europarl/fr/ep-02-11-18-fr.txt', 'data/europarl/fr/ep-00-06-13-fr.txt', 'data/europarl/fr/ep-02-03-20-fr.txt', 'data/europarl/fr/ep-00-04-10-fr.txt', 'data/europarl/fr/ep-05-05-25-fr.txt', 'data/europarl/fr/ep-05-11-17-fr.txt', 'data/europarl/fr/ep-01-07-02-fr.txt', 'data/europarl/fr/ep-04-10-27-fr.txt', 'd

## Mise à jour des fonctions pour utiliser word_tokenize

In [64]:
! pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 914.8 kB/s eta 0:00:00a 0:00:01


In [65]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/usmanalfayed/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [66]:
import nltk

def lire_fichier(chemin):
    with open(chemin, 'r', encoding='utf-8') as f:
        return f.read()

def tokenizer_nltk(chaine):
    # Utiliser word_tokenize de NLTK pour découper les mots
    return nltk.word_tokenize(chaine, language='french')

In [67]:
def decouper_requete(requete):
    # Utiliser word_tokenize de NLTK pour découper les mots
    mots_requete = nltk.word_tokenize(requete, language='french')
    mots_requete = [mot.lower() for mot in mots_requete]
    print(f"Mots de la requête : {mots_requete}")
    return mots_requete

In [68]:
import re

def afficher_contextes(chaine, terme, taille_contexte=30):
    matches = re.finditer(re.escape(terme), chaine)
    contexts = []
    for match in matches:
        gauche = max(match.start() - taille_contexte, 0)
        droite = match.end() + taille_contexte
        contexts.append(chaine[gauche:droite])
    return contexts

def chercher_documents_et_contextes(index, mots_requete, taille_contexte=30):
    documents_trouves = {}
    contextes = {}
    for mot in mots_requete:
        if mot in index:
            print(f"Le mot '{mot}' est présent dans l'index et est trouvé dans les documents suivants : {index[mot]}")
            for doc in index[mot]:
                if doc not in documents_trouves:
                    documents_trouves[doc] = 0
                    contextes[doc] = {}
                documents_trouves[doc] += 1
                # Lire le fichier et trouver les contextes
                chaine = lire_fichier(doc)
                if mot not in contextes[doc]:
                    contextes[doc][mot] = []
                contextes[doc][mot].extend(afficher_contextes(chaine, mot, taille_contexte))
        else:
            print(f"Le mot '{mot}' n'est pas présent dans l'index.")
    return documents_trouves, contextes

def afficher_contextes_documents(contextes):
    if not contextes:
        print("Aucun contexte trouvé pour les mots de la requête.")
    else:
        print("Contextes trouvés :")
        for doc, contextes_mots in contextes.items():
            print(f"\nDocument: {doc}")
            for mot, contextes_list in contextes_mots.items():
                print(f"\nMot: {mot}")
                for contexte in contextes_list:
                    print(f"...{contexte}...")

In [69]:
# Exemple d'utilisation
index = charger_index("index.json")

if index:
    requete = "Commission Européenne"
    mots_requete = decouper_requete(requete)
    documents_trouves, contextes = chercher_documents_et_contextes(index, mots_requete)
    afficher_contextes_documents(contextes)

Index chargé avec succès.
Mots de la requête : ['commission', 'européenne']
Le mot 'commission' est présent dans l'index et est trouvé dans les documents suivants : ['data/europarl/fr/ep-04-07-22-fr.txt', 'data/europarl/fr/ep-01-09-06-fr.txt', 'data/europarl/fr/ep-00-05-03-fr.txt', 'data/europarl/fr/ep-02-05-14-fr.txt', 'data/europarl/fr/ep-02-03-11-fr.txt', 'data/europarl/fr/ep-03-10-08-fr.txt', 'data/europarl/fr/ep-03-06-03-fr.txt', 'data/europarl/fr/ep-04-12-16-fr.txt', 'data/europarl/fr/ep-02-11-20-fr.txt', 'data/europarl/fr/ep-03-06-19-fr.txt', 'data/europarl/fr/ep-01-12-17-fr.txt', 'data/europarl/fr/ep-02-04-24-fr.txt', 'data/europarl/fr/ep-03-04-08-fr.txt', 'data/europarl/fr/ep-01-10-23-fr.txt', 'data/europarl/fr/ep-02-11-18-fr.txt', 'data/europarl/fr/ep-00-06-13-fr.txt', 'data/europarl/fr/ep-02-03-20-fr.txt', 'data/europarl/fr/ep-00-04-10-fr.txt', 'data/europarl/fr/ep-05-05-25-fr.txt', 'data/europarl/fr/ep-05-11-17-fr.txt', 'data/europarl/fr/ep-01-07-02-fr.txt', 'data/europarl/

## INDEX ET INDEX INVERSÉ

In [70]:
import glob
import nltk
import json

# Assurez-vous que NLTK est téléchargé
nltk.download('punkt')

def lire_fichier(chemin):
    with open(chemin, 'r', encoding='utf-8') as f:
        return f.read()

def tokenizer_nltk(chaine):
    # Utiliser word_tokenize de NLTK pour découper les mots
    return nltk.word_tokenize(chaine, language='french')

def creer_index(chemin_base):
    index = {}
    for chemin in glob.glob(chemin_base):
        chaine = lire_fichier(chemin)
        mots = tokenizer_nltk(chaine)
        for mot in mots:
            mot = mot.lower()
            if mot not in index:
                index[mot] = set()  # Utiliser un ensemble pour éviter les doublons
            index[mot].add(chemin)
    return index

def creer_index_inverse(index):
    index_inverse = {}
    for mot, documents in index.items():
        for doc in documents:
            if doc not in index_inverse:
                index_inverse[doc] = set()
            index_inverse[doc].add(mot)
    return index_inverse

def charger_index(fichier):
    try:
        with open(fichier, 'r', encoding='utf-8') as f:
            index = json.load(f)
        print("Index chargé avec succès.")
        return index
    except FileNotFoundError:
        print(f"Le fichier '{fichier}' n'a pas été trouvé.")
        return None
    except json.JSONDecodeError:
        print(f"Erreur lors du chargement du fichier '{fichier}'. Assurez-vous qu'il est correctement formaté.")
        return None

def requeter_documents(requete, index):
    mots_requete = tokenizer_nltk(requete)
    mots_requete = [mot.lower() for mot in mots_requete]
    documents_trouves = set()
    for mot in mots_requete:
        if mot in index:
            if not documents_trouves:
                documents_trouves = set(index[mot])
            else:
                documents_trouves.intersection_update(index[mot])
    return documents_trouves

# Chemin de base pour les fichiers à indexer
chemin_base = "data/europarl/fr/*"

# Créer les index
index = creer_index(chemin_base)
index_inverse = creer_index_inverse(index)

# Afficher les statistiques des index
print("Nombre de termes différents : ", len(index.keys()))
print("Nombre de documents :", len(index_inverse.keys()))

# Stocker les index
with open("index.json", "w") as w:
    w.write(json.dumps({mot: list(docs) for mot, docs in index.items()}))

with open("index_inverse.json", "w") as w:
    w.write(json.dumps({doc: list(mots) for doc, mots in index_inverse.items()}))

# Charger l'index
index = charger_index("index.json")

if index:
    # Requête 1
    requete = "liberté humaine"
    docs_trouves = requeter_documents(requete, index)
    print(f"Nombre de documents trouvés pour '{requete}':", len(docs_trouves))
    
    # Requête 2
    requete = "sensibilisation minorités"
    docs_trouves = requeter_documents(requete, index)
    print(f"Nombre de documents trouvés pour '{requete}':", len(docs_trouves))

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/usmanalfayed/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Nombre de termes différents :  112515
Nombre de documents : 363
Index chargé avec succès.
Nombre de documents trouvés pour 'liberté humaine': 311
Nombre de documents trouvés pour 'sensibilisation minorités': 133


## Calcul TF IDF

In [71]:
import glob
import json
import nltk
from collections import defaultdict
import math

nltk.download('punkt')

def lire_fichier(chemin):
    with open(chemin, 'r', encoding='utf-8') as f:
        return f.read()

def tokenizer_nltk(chaine):
    # Utiliser word_tokenize de NLTK pour découper les mots
    return nltk.word_tokenize(chaine, language='french')

def creer_index(chemin_base):
    index = defaultdict(set)
    all_docs = {}
    for chemin in glob.glob(chemin_base):
        chaine = lire_fichier(chemin)
        mots = tokenizer_nltk(chaine)
        mot_counts = defaultdict(int)
        for mot in mots:
            mot = mot.lower()
            index[mot].add(chemin)
            mot_counts[mot] += 1
        total_mots = sum(mot_counts.values())
        tf = {mot: count / total_mots for mot, count in mot_counts.items()}
        all_docs[chemin] = tf
    return index, all_docs

def calculer_idf(index, total_docs):
    idf = {}
    for mot, docs in index.items():
        idf[mot] = math.log10(total_docs / len(docs))
    return idf

# Chemin de base pour les fichiers à indexer
chemin_base = "data/europarl/fr/*"

# Créer les index et calculer les TF
index, all_docs = creer_index(chemin_base)

# Calculer l'IDF
total_docs = len(all_docs)
idf = calculer_idf(index, total_docs)

# Afficher les résultats
print("TF pour chaque document :")
for doc, tf in all_docs.items():
    print(f"{doc}: {tf}")

print("\nIDF pour chaque mot :")
for mot, score in idf.items():
    print(f"{mot}: {score}")

# Stocker les résultats
with open("tf.json", "w") as w:
    w.write(json.dumps(all_docs))

with open("idf.json", "w") as w:
    w.write(json.dumps(idf))

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/usmanalfayed/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


TF pour chaque document :
data/europarl/fr/ep-02-04-25-fr.txt: {'flottes': 0.0004453080722208728, 'de': 0.053517933770544895, 'pêche': 0.002024127601003967, "l'ordre": 0.00020241276010039673, 'du': 0.008946643996437536, 'jour': 0.00048579062424095217, 'appelle': 0.00012144765606023804, 'le': 0.02088899684236094, 'rapport': 0.003279086713626427, '(': 0.001416889320702777, 'a5-0092/2002': 4.048255202007934e-05, ')': 0.001416889320702777, 'm.': 0.0012549591126224597, 'kindermann': 0.0006882033843413489, ',': 0.05189863168974172, 'au': 0.006558173427252854, 'nom': 0.00032386041616063474, 'la': 0.033721965832726096, 'commission': 0.003967290097967776, 'sur': 0.005627074730791029, 'annuel': 0.00016193020808031737, 'conseil': 0.0017812322888834911, 'et': 0.021293822362561735, 'parlement': 0.001538336976763015, 'européen': 0.0012144765606023804, 'les': 0.02044368877014007, 'résultats': 0.0002833778641405554, 'des': 0.019715002833778642, 'programmes': 0.00048579062424095217, "d'orientation": 0.

## Verification des resultats

In [73]:
import json
import math
import nltk

# Assurez-vous que NLTK est téléchargé
nltk.download('punkt')

def charger_fichier(fichier):
    try:
        with open(fichier, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"Fichier chargé avec succès depuis {fichier}.")
        return data
    except FileNotFoundError:
        print(f"Le fichier '{fichier}' n'a pas été trouvé.")
        return None
    except json.JSONDecodeError:
        print(f"Erreur lors du chargement du fichier '{fichier}'. Assurez-vous qu'il est correctement formaté.")
        return None

# Charger les index et TF/IDF depuis les fichiers JSON
all_docs = charger_fichier("tf.json")
idf = charger_fichier("idf.json")
index = charger_fichier("index.json")

# Vérification des TF
print("\nVérification des TF pour chaque document :")
tf_correct = True
for doc, tf in all_docs.items():
    somme_tf = sum(tf.values())
    if not (0.99 <= somme_tf <= 1.01):
        tf_correct = False
    print(f"{doc}: Somme des TF = {somme_tf:.4f} (doit être proche de 1.0)")

if tf_correct:
    print("\nToutes les sommes des TF sont correctes.")
else:
    print("\nAttention : Certaines sommes des TF ne sont pas proches de 1.0.")

# Vérification des IDF
print("\nVérification des IDF pour chaque mot :")
idf_correct = True
total_docs = len(all_docs)
for mot, docs in index.items():
    expected_idf = math.log10(total_docs / len(docs))
    if abs(idf[mot] - expected_idf) > 0.0001:
        idf_correct = False
    print(f"{mot}: Calculé = {idf[mot]:.5f}, Attendu = {expected_idf:.5f}")

if idf_correct:
    print("\nTous les IDF sont corrects.")
else:
    print("\nAttention : Certains IDF ne sont pas corrects.")

# Exemple d'utilisation des index pour les requêtes
def tokenizer_nltk(chaine):
    return nltk.word_tokenize(chaine, language='french')

def requeter_documents(requete, index):
    mots_requete = tokenizer_nltk(requete)
    mots_requete = [mot.lower() for mot in mots_requete]
    documents_trouves = set()
    for mot in mots_requete:
        if mot in index:
            if not documents_trouves:
                documents_trouves = set(index[mot])
            else:
                documents_trouves.intersection_update(index[mot])
    return documents_trouves

if index:
    # Requête 1
    requete = "liberté humaine"
    docs_trouves = requeter_documents(requete, index)
    print(f"\nNombre de documents trouvés pour '{requete}':", len(docs_trouves))
    
    # Requête 2
    requete = "sensibilisation minorités"
    docs_trouves = requeter_documents(requete, index)
    print(f"\nNombre de documents trouvés pour '{requete}':", len(docs_trouves))

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/usmanalfayed/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Fichier chargé avec succès depuis tf.json.
Fichier chargé avec succès depuis idf.json.
Fichier chargé avec succès depuis index.json.

Vérification des TF pour chaque document :
data/europarl/fr/ep-02-04-25-fr.txt: Somme des TF = 1.0000 (doit être proche de 1.0)
data/europarl/fr/ep-03-01-14-fr.txt: Somme des TF = 1.0000 (doit être proche de 1.0)
data/europarl/fr/ep-00-09-20-fr.txt: Somme des TF = 1.0000 (doit être proche de 1.0)
data/europarl/fr/ep-00-02-14-fr.txt: Somme des TF = 1.0000 (doit être proche de 1.0)
data/europarl/fr/ep-04-01-14-fr.txt: Somme des TF = 1.0000 (doit être proche de 1.0)
data/europarl/fr/ep-04-12-02-fr.txt: Somme des TF = 1.0000 (doit être proche de 1.0)
data/europarl/fr/ep-04-02-25-fr.txt: Somme des TF = 1.0000 (doit être proche de 1.0)
data/europarl/fr/ep-03-09-03-fr.txt: Somme des TF = 1.0000 (doit être proche de 1.0)
data/europarl/fr/ep-05-10-25-fr.txt: Somme des TF = 1.0000 (doit être proche de 1.0)
data/europarl/fr/ep-03-06-02-fr.txt: Somme des TF = 1.0000

## Seuil de similarité cosinus

In [74]:
import json
import math
import nltk
from collections import defaultdict

# Assurez-vous que NLTK est téléchargé
nltk.download('punkt')

def charger_fichier(fichier):
    try:
        with open(fichier, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"Fichier chargé avec succès depuis {fichier}.")
        return data
    except FileNotFoundError:
        print(f"Le fichier '{fichier}' n'a pas été trouvé.")
        return None
    except json.JSONDecodeError:
        print(f"Erreur lors du chargement du fichier '{fichier}'. Assurez-vous qu'il est correctement formaté.")
        return None

# Charger les index et TF/IDF depuis les fichiers JSON
all_docs = charger_fichier("tf.json")
idf = charger_fichier("idf.json")
index = charger_fichier("index.json")

# Fonction pour tokenizer la requête
def tokenizer_nltk(chaine):
    return nltk.word_tokenize(chaine, language='french')

# Fonction pour indexer la requête
def indexer_requete(requete, index, idf):
    mots_requete = tokenizer_nltk(requete)
    mots_requete = [mot.lower() for mot in mots_requete]
    requete_indexee = {}
    total_mots = len(mots_requete)
    mot_counts = defaultdict(int)
    for mot in mots_requete:
        mot_counts[mot] += 1
    for mot, count in mot_counts.items():
        if mot in idf:
            requete_indexee[mot] = (count / total_mots) * idf[mot]
    return requete_indexee

# Fonction pour calculer la similarité cosinus
def calculer_sim_cosinus(vecA, vecB):
    dot_product = sum(vecA.get(mot, 0) * vecB.get(mot, 0) for mot in vecA)
    normA = math.sqrt(sum(val ** 2 for val in vecA.values()))
    normB = math.sqrt(sum(val ** 2 for val in vecB.values()))
    if normA == 0 or normB == 0:
        return 0.0
    return dot_product / (normA * normB)

# Fonction pour calculer la pondération cosinus
def calculer_ponderation_cosinus(requete_indexee, index, all_docs, docs_trouves):
    scores = {}
    for doc in docs_trouves:
        vec_doc = all_docs[doc]
        score = calculer_sim_cosinus(requete_indexee, vec_doc)
        scores[doc] = score
    return scores

# Exemple d'utilisation des index pour les requêtes
def requeter_documents(requete, index):
    mots_requete = tokenizer_nltk(requete)
    mots_requete = [mot.lower() for mot in mots_requete]
    documents_trouves = set()
    for mot in mots_requete:
        if mot in index:
            if not documents_trouves:
                documents_trouves = set(index[mot])
            else:
                documents_trouves.intersection_update(index[mot])
    return documents_trouves

if index and idf and all_docs:
    # Requête
    requete = "liberté humaine"
    
    # Indexer la requête
    requete_indexee = indexer_requete(requete, index, idf)
    
    # Trouver les documents correspondants à la requête
    docs_trouves = requeter_documents(requete, index)
    
    # Calculer la pondération cosinus pour les documents trouvés
    scores = calculer_ponderation_cosinus(requete_indexee, index, all_docs, docs_trouves)
    
    # Afficher les scores
    print("\nScores des documents trouvés pour la requête 'liberté humaine':")
    for doc, score in sorted(scores.items(), key=lambda item: item[1], reverse=True):
        print(f"{doc}: {score:.4f}")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/usmanalfayed/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Fichier chargé avec succès depuis tf.json.
Fichier chargé avec succès depuis idf.json.
Fichier chargé avec succès depuis index.json.

Scores des documents trouvés pour la requête 'liberté humaine':
data/europarl/fr/ep-01-11-29-fr.txt: 0.0175
data/europarl/fr/ep-00-03-30-fr.txt: 0.0063
data/europarl/fr/ep-02-09-23-fr.txt: 0.0050
data/europarl/fr/ep-01-09-12-fr.txt: 0.0044
data/europarl/fr/ep-00-04-10-fr.txt: 0.0039
data/europarl/fr/ep-02-09-05-fr.txt: 0.0039
data/europarl/fr/ep-00-09-21-fr.txt: 0.0038
data/europarl/fr/ep-03-04-09-fr.txt: 0.0037
data/europarl/fr/ep-05-01-27-fr.txt: 0.0035
data/europarl/fr/ep-01-02-01-fr.txt: 0.0034
data/europarl/fr/ep-05-04-11-fr.txt: 0.0034
data/europarl/fr/ep-01-05-17-fr.txt: 0.0032
data/europarl/fr/ep-02-07-04-fr.txt: 0.0032
data/europarl/fr/ep-02-10-24-fr.txt: 0.0032
data/europarl/fr/ep-04-05-03-fr.txt: 0.0031
data/europarl/fr/ep-04-09-16-fr.txt: 0.0031
data/europarl/fr/ep-04-03-31-fr.txt: 0.0030
data/europarl/fr/ep-02-04-11-fr.txt: 0.0030
data/europ

## Classement par pertinence(Ranking)

In [78]:
# Exemple d'utilisation des index pour les requêtes
def requeter_documents(requete, index):
    mots_requete = tokenizer_nltk(requete)
    mots_requete = [mot.lower() for mot in mots_requete]
    documents_trouves = set()
    for mot in mots_requete:
        if mot in index:
            if not documents_trouves:
                documents_trouves = set(index[mot])
            else:
                documents_trouves.intersection_update(index[mot])
    return documents_trouves

def afficher_documents_classement(requete, index, idf, all_docs):
    # Indexer la requête
    requete_indexee = indexer_requete(requete, index, idf)
    
    # Trouver les documents correspondants à la requête
    docs_trouves = requeter_documents(requete, index)
    
    # Calculer la pondération cosinus pour les documents trouvés
    scores = calculer_ponderation_cosinus(requete_indexee, index, all_docs, docs_trouves)
    
    # Trier les documents par pertinence
    docs_trouves_pond_liste = [[sim, chemin] for chemin, sim in scores.items()]
    docs_trouves_pond_liste = sorted(docs_trouves_pond_liste, reverse=True, key=lambda x: x[0])
    
    # Afficher les documents classés par pertinence
    print(f"\nClassement des documents trouvés pour la requête '{requete}':")
    for sim, chemin in docs_trouves_pond_liste:
        print(f"{sim:.6f} {chemin.split('/')[-1]}")

# Exemple d'utilisation
if index and idf and all_docs:
    requete = "liberté humaine"
    afficher_documents_classement(requete, index, idf, all_docs)


Classement des documents trouvés pour la requête 'liberté humaine':
0.017527 ep-01-11-29-fr.txt
0.006318 ep-00-03-30-fr.txt
0.004969 ep-02-09-23-fr.txt
0.004378 ep-01-09-12-fr.txt
0.003909 ep-00-04-10-fr.txt
0.003872 ep-02-09-05-fr.txt
0.003818 ep-00-09-21-fr.txt
0.003665 ep-03-04-09-fr.txt
0.003458 ep-05-01-27-fr.txt
0.003396 ep-01-02-01-fr.txt
0.003391 ep-05-04-11-fr.txt
0.003182 ep-01-05-17-fr.txt
0.003169 ep-02-07-04-fr.txt
0.003165 ep-02-10-24-fr.txt
0.003085 ep-04-05-03-fr.txt
0.003067 ep-04-09-16-fr.txt
0.002997 ep-04-03-31-fr.txt
0.002974 ep-02-04-11-fr.txt
0.002872 ep-04-01-29-fr.txt
0.002830 ep-05-09-26-fr.txt
0.002828 ep-01-07-05-fr.txt
0.002790 ep-01-04-04-fr.txt
0.002768 ep-01-09-06-fr.txt
0.002731 ep-00-02-02-fr.txt
0.002705 ep-02-07-02-fr.txt
0.002640 ep-03-11-17-fr.txt
0.002561 ep-03-04-10-fr.txt
0.002521 ep-05-05-11-fr.txt
0.002500 ep-03-03-13-fr.txt
0.002499 ep-01-11-14-fr.txt
0.002445 ep-03-09-03-fr.txt
0.002445 ep-03-11-20-fr.txt
0.002438 ep-03-10-09-fr.txt
0.00238

# Structuration de l’espace de recherche

### 1. KD-Tree

In [79]:
! pip install scipy

In [84]:
import numpy as np
from scipy.spatial import KDTree
import time

# Convertir les vecteurs de documents en format numpy array pour KD-Tree
def construire_kdtree(all_docs):
    docs_vectors = []
    docs_keys = []
    for doc, tfidf in all_docs.items():
        vector = [tfidf.get(mot, 0) for mot in idf.keys()]
        docs_vectors.append(vector)
        docs_keys.append(doc)
    kdtree = KDTree(docs_vectors)
    return kdtree, docs_keys

# Fonction pour utiliser le KD-Tree pour la recherche de documents
def rechercher_kdtree(kdtree, docs_keys, requete_indexee):
    requete_vector = [requete_indexee.get(mot, 0) for mot in idf.keys()]
    distances, indices = kdtree.query(requete_vector, k=len(docs_keys))
    docs_trouves = [docs_keys[i] for i in indices]
    return docs_trouves

# Exemple d'utilisation et mesure de performance du KD-Tree
if index and idf and all_docs:
    # Construire le KD-Tree
    start_time = time.time()
    kdtree, docs_keys = construire_kdtree(all_docs)
    construction_time = time.time() - start_time
    
    # Requête
    requete = "liberté humaine"
    requete_indexee = indexer_requete(requete, index, idf)
    
    # Recherche avec le KD-Tree
    start_time = time.time()
    docs_trouves = rechercher_kdtree(kdtree, docs_keys, requete_indexee)
    recherche_time = time.time() - start_time
    
    # Afficher les résultats
    scores = calculer_ponderation_cosinus(requete_indexee, index, all_docs, docs_trouves)
    docs_trouves_pond_liste = [[sim, chemin] for chemin, sim in scores.items()]
    docs_trouves_pond_liste = sorted(docs_trouves_pond_liste, reverse=True, key=lambda x: x[0])
    
    print(f"\nRésultats de recherche avec KD-Tree pour la requête '{requete}':")
    for sim, chemin in docs_trouves_pond_liste[:10]:
        print(f"{sim:.6f} {chemin.split('/')[-1]}")
    
    print(f"\nTemps de construction du KD-Tree: {construction_time:.4f} secondes")
    print(f"Temps de recherche avec KD-Tree: {recherche_time:.4f} secondes")


Résultats de recherche avec KD-Tree pour la requête 'liberté humaine':
0.017527 ep-01-11-29-fr.txt
0.006318 ep-00-03-30-fr.txt
0.004969 ep-02-09-23-fr.txt
0.004378 ep-01-09-12-fr.txt
0.003909 ep-00-04-10-fr.txt
0.003872 ep-02-09-05-fr.txt
0.003818 ep-00-09-21-fr.txt
0.003665 ep-03-04-09-fr.txt
0.003458 ep-05-01-27-fr.txt
0.003396 ep-01-02-01-fr.txt

Temps de construction du KD-Tree: 5.4462 secondes
Temps de recherche avec KD-Tree: 0.0338 secondes


### 2. K-Means

In [81]:
! pip install scikit-learn

In [85]:
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
import time

# Construire K-Means
def construire_kmeans(all_docs, n_clusters=10):
    docs_vectors = []
    docs_keys = []
    for doc, tfidf in all_docs.items():
        vector = [tfidf.get(mot, 0) for mot in idf.keys()]
        docs_vectors.append(vector)
        docs_keys.append(doc)
    kmeans = KMeans(n_clusters=n_clusters, random_state=0).fit(docs_vectors)
    return kmeans, docs_vectors, docs_keys

# Fonction pour utiliser K-Means pour la recherche de documents
def rechercher_kmeans(kmeans, docs_vectors, docs_keys, requete_indexee):
    requete_vector = [requete_indexee.get(mot, 0) for mot in idf.keys()]
    cluster = kmeans.predict([requete_vector])[0]
    cluster_docs = [docs_keys[i] for i in range(len(docs_vectors)) if kmeans.labels_[i] == cluster]
    return cluster_docs

# Exemple d'utilisation et mesure de performance de K-Means
if index and idf and all_docs:
    # Construire K-Means
    start_time = time.time()
    kmeans, docs_vectors, docs_keys = construire_kmeans(all_docs)
    construction_time = time.time() - start_time
    
    # Requête
    requete = "liberté humaine"
    requete_indexee = indexer_requete(requete, index, idf)
    
    # Recherche avec K-Means
    start_time = time.time()
    docs_trouves = rechercher_kmeans(kmeans, docs_vectors, docs_keys, requete_indexee)
    recherche_time = time.time() - start_time
    
    # Afficher les résultats
    scores = calculer_ponderation_cosinus(requete_indexee, index, all_docs, docs_trouves)
    docs_trouves_pond_liste = [[sim, chemin] for chemin, sim in scores.items()]
    docs_trouves_pond_liste = sorted(docs_trouves_pond_liste, reverse=True, key=lambda x: x[0])
    
    print(f"\nRésultats de recherche avec K-Means pour la requête '{requete}':")
    for sim, chemin in docs_trouves_pond_liste[:10]:
        print(f"{sim:.6f} {chemin.split('/')[-1]}")
    
    print(f"\nTemps de construction de K-Means: {construction_time:.4f} secondes")
    print(f"Temps de recherche avec K-Means: {recherche_time:.4f} secondes")


Résultats de recherche avec K-Means pour la requête 'liberté humaine':
0.006318 ep-00-03-30-fr.txt
0.003182 ep-01-05-17-fr.txt
0.002828 ep-01-07-05-fr.txt
0.002768 ep-01-09-06-fr.txt
0.002390 ep-00-05-19-fr.txt
0.002136 ep-01-02-15-fr.txt
0.002133 ep-00-03-15-fr.txt
0.002030 ep-00-02-17-fr.txt
0.001554 ep-02-02-04-fr.txt
0.001530 ep-02-02-07-fr.txt

Temps de construction de K-Means: 10.1078 secondes
Temps de recherche avec K-Means: 0.0136 secondes


### 3. Locality-Sensitive Hashing (LSH)

In [86]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import time

# Construire LSH
def construire_lsh(all_docs, n_hashes=10):
    vectorizer = TfidfVectorizer()
    docs_vectors = []
    docs_keys = []
    for doc, tfidf in all_docs.items():
        vector = [tfidf.get(mot, 0) for mot in idf.keys()]
        docs_vectors.append(vector)
        docs_keys.append(doc)
    lsh = NearestNeighbors(n_neighbors=n_hashes, algorithm='ball_tree').fit(docs_vectors)
    return lsh, docs_keys

# Fonction pour utiliser LSH pour la recherche de documents
def rechercher_lsh(lsh, docs_keys, requete_indexee):
    requete_vector = [requete_indexee.get(mot, 0) for mot in idf.keys()]
    distances, indices = lsh.kneighbors([requete_vector])
    docs_trouves = [docs_keys[i] for i in indices[0]]
    return docs_trouves

# Exemple d'utilisation et mesure de performance de LSH
if index and idf and all_docs:
    # Construire LSH
    start_time = time.time()
    lsh, docs_keys = construire_lsh(all_docs)
    construction_time = time.time() - start_time
    
    # Requête
    requete = "liberté humaine"
    requete_indexee = indexer_requete(requete, index, idf)
    
    # Recherche avec LSH
    start_time = time.time()
    docs_trouves = rechercher_lsh(lsh, docs_keys, requete_indexee)
    recherche_time = time.time() - start_time
    
    # Afficher les résultats
    scores = calculer_ponderation_cosinus(requete_indexee, index, all_docs, docs_trouves)
    docs_trouves_pond_liste = [[sim, chemin] for chemin, sim in scores.items()]
    docs_trouves_pond_liste = sorted(docs_trouves_pond_liste, reverse=True, key=lambda x: x[0])
    
    print(f"\nRésultats de recherche avec LSH pour la requête '{requete}':")
    for sim, chemin in docs_trouves_pond_liste[:10]:
        print(f"{sim:.6f} {chemin.split('/')[-1]}")
    
    print(f"\nTemps de construction de LSH: {construction_time:.4f} secondes")
    print(f"Temps de recherche avec LSH: {recherche_time:.4f} secondes")


Résultats de recherche avec LSH pour la requête 'liberté humaine':
0.003909 ep-00-04-10-fr.txt
0.001758 ep-02-12-17-fr.txt
0.001668 ep-01-10-04-fr.txt
0.001261 ep-01-06-11-fr.txt
0.001024 ep-03-06-30-fr.txt
0.001022 ep-00-07-07-fr.txt
0.000661 ep-02-02-05-fr.txt
0.000639 ep-00-03-29-fr.txt
0.000374 ep-00-05-15-fr.txt
0.000292 ep-02-06-10-fr.txt

Temps de construction de LSH: 5.4510 secondes
Temps de recherche avec LSH: 0.0649 secondes


## Comparaison des Résultats



### Analyse des Résultats

1. **KD-Tree** :
   - Le temps de construction est moyen (5.4462 secondes).
   - Le temps de recherche est très rapide (0.0338 secondes).
   - Les scores de similarité sont les plus élevés comparés aux autres méthodes, ce qui suggère une bonne précision.

2. **K-Means** :
   - Le temps de construction est le plus long (10.1078 secondes).
   - Le temps de recherche est très rapide (0.0136 secondes).
   - Les scores de similarité sont plus bas que ceux obtenus avec KD-Tree, ce qui pourrait indiquer une précision légèrement moindre.

3. **LSH** :
   - Le temps de construction est similaire à celui du KD-Tree (5.4510 secondes).
   - Le temps de recherche est le plus long parmi les trois méthodes (0.0649 secondes).
   - Les scores de similarité sont également plus bas, ce qui pourrait indiquer une précision moindre comparée à KD-Tree.

### Conclusion

- **KD-Tree** semble offrir un bon compromis entre le temps de construction et la précision des résultats. C'est une méthode efficace pour des bases de données de taille modérée.
- **K-Means** a le temps de construction le plus long, mais offre un temps de recherche très rapide. Cependant, la précision semble légèrement inférieure à celle du KD-Tree.
- **LSH** a un temps de construction comparable à KD-Tree, mais un temps de recherche plus long et une précision légèrement inférieure.

